In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import PIL

In [2]:
import tensorflow as tf
print(tf.__version__)

2026-02-15 06:26:48.898197: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771136809.073975      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771136809.124718      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771136809.548970      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771136809.549011      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771136809.549014      55 computation_placer.cc:177] computation placer alr

2.19.0


In [ ]:
file_path = r"/kaggle/input/plantvillage-dataset/color"

IMAGE_SIZE = (224,224)
BATCH_SIZE = 16
SEED = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    directory = file_path,
    validation_split = 0.3,
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE,
    seed = SEED,
    subset ="training"
    
) 

temp_ds = tf.keras.utils.image_dataset_from_directory(
    directory = file_path,
    validation_split = 0.3,
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE,
    seed = SEED,
    subset ="validation"
    
)


val_ds = temp_ds.take(int(0.5 * len(temp_ds)))
test_ds = temp_ds.skip(int(0.5 * len(temp_ds)))

In [ ]:
print(len(train_ds))

In [ ]:
class_name = train_ds.class_names
len(class_name)
class_name

In [ ]:
import json
with open("Disease_Name.json","w") as f:
    json.dump(class_name,f)
    

In [ ]:
type(train_ds)

In [ ]:
train_ds = train_ds.shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.shuffle(1000).prefetch(tf.data.AUTOTUNE)

In [ ]:
# plt.figure(figsize=(10,10))
# for images , labels in train_ds.take(1):
#     for i in range(9):
#         plt.subplot(3,3,i+1)
#         plt.axis("off")
#         plt.imshow(images[i].numpy().astype("uint8"))
#         plt.title(class_name[labels[i]])

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model, layers, Sequential 
from tensorflow.keras import layers

In [ ]:
# Create CNN Model 

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])

model_cnn = tf.keras.Sequential([

    layers.Input(shape=(256,256,3)),
    layers.Rescaling(1./255),
    data_augmentation,

    layers.Conv2D(32,kernel_size=(3,3), padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64,kernel_size=(3,3), padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128,kernel_size=(3,3), padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # layers.Conv2D(256,kernel_size=(3,3), padding='same',activation='relu'),
    # layers.BatchNormalization(),
    # layers.MaxPooling2D(),

    layers.Dropout(0.2),

    layers.GlobalAveragePooling2D(),

    #Dense layers
    layers.Dense(128,activation ='relu'),
    layers.Dropout(0.3),

    layers.Dense(38,activation = "softmax")
])

model_cnn.summary()

In [ ]:
model_cnn.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.003),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics =['accuracy']
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor ='val_loss',
    patience = 2,
    restore_best_weights= True
)

model_cnn.fit(train_ds,validation_data = val_ds , epochs=30, callbacks=[early_stopping])

In [ ]:
model_cnn.evaluate(test_ds)

In [ ]:
model_cnn.save("Plant_Cnn_Model.keras")

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input



In [ ]:
# Transfer Learning

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet"
)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2)
])
base_model.trainable = False
inputs = layers.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x , training = False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128 , activation = 'relu',
                kernel_regularizer = tf.keras.regularizers.l2(1e-3))(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(38 , activation = "softmax")(x)

model_effi = Model(inputs,outputs)

model_effi.summary()

In [ ]:
model_effi.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.003),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics =['accuracy']
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor ='val_loss',
    patience = 2,
    restore_best_weights= True
)

model_effi.fit(train_ds,validation_data = val_ds , epochs=30, callbacks=[early_stopping])

In [ ]:
model_effi.evaluate(test_ds)

In [ ]:
# Now Time To Fine Tuning
base_model.trainable = True
for layers in base_model.layers[:-30]:
    layers.trainable =False

model_effi.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics =['accuracy']
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor ='val_loss',
    patience = 2,
    restore_best_weights= True
)

history = model_effi.fit(train_ds,validation_data = val_ds , epochs=30, callbacks=[early_stopping])

In [ ]:
model_effi.evaluate(test_ds)

In [ ]:
model_effi.save("Plant_effi_Model.keras")